# Tâche 2 : Question-réponse avec poursuite du pré-entraînement du modèle génératif

Cette tâche consiste à poursuivre le préentraînement d'un modèle pré‑entraîné sur un corpus de petite taille. On vous demande d'évaluer la qualité des réponses du modèle après avoir complété l'entraînement supplémentaire. L'évaluation du modèle est effectué avec les mêmes paires **questions-réponses** que la tâche 1. 

NOTE: Il est important de sauvegarder le modèle pré-entraîné dans cette tâche (ainsi que le tokenizer) car vous les réutilisez à la tâche 3. 

> Il est recommandé de faire ce travail pratique en utilisant une carte graphique GPU compatible avec HuggingFace/Pytorch.
> Si votre machine n’en possède pas, vous pouvez utiliser **Google Colab** pour exécuter le *notebook* dans le cloud.

In [1]:
!pip install transformers
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

Looking in indexes: https://download.pytorch.org/whl/cu132


In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
torch.cuda.is_available()

True

## 1. Chargement du modèle Hugging Face et du tokenizer

In [23]:
model_name = "EleutherAI/pythia-410m"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir="saved_models/pythia-410m")
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir="saved_models/pythia-410m")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 1040.81it/s]


In [5]:
# test
inputs = tokenizer("Hello, I am", return_tensors="pt", max_length=24)
tokens = model.generate(**inputs)
print(tokenizer.decode(tokens[0]))

C:\Users\Heros\PycharmProjects\IFT-7022-travail-2\.venv\Lib\site-packages\transformers\generation\utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Hello, I am a newbie to this site and I am trying to get a list of all the files in the


## 2. Fonctions de test question-réponse

## 3. Poursuite du pré-entraînement du modèle

In [6]:
dataset = load_dataset("json", data_files="data/train_texts.json")

In [7]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [8]:
training_args = TrainingArguments(
    output_dir="checkpoints/pythia-410m-finetuned-t2",
    num_train_epochs=1,
    per_device_train_batch_size=8,   # Ajuster selon la VRAM disponible
    gradient_accumulation_steps=1,
    learning_rate=1e-6,
    weight_decay=0.1,
    save_steps=20,
    logging_steps=5,
    fp16=False,    # Doit être désactivé sur RTX 4070 pour utiliser bf16
    bf16=True,
    max_grad_norm=1.0
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

In [9]:
trainer.train()

trainer.save_model("saved_models/pythia-410m-final-t2")
tokenizer.save_pretrained(".saved_models/pythia-410m-final-t2")

Step,Training Loss
5,3.375555
10,3.440677
15,3.573855
20,3.404184
25,3.450799
30,3.615626
35,3.454491
40,3.659105
45,3.563745
50,3.571819


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it]


('.saved_models/pythia-410m-final-t2\\tokenizer_config.json',
 '.saved_models/pythia-410m-final-t2\\tokenizer.json')

## 4. Génération de réponses avec le nouveau modèle

In [26]:
model = AutoModelForCausalLM.from_pretrained("saved_models/pythia-410m-final-t2")
tokenizer = AutoTokenizer.from_pretrained("saved_models/pythia-410m-final-t2")

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 1196.90it/s]


In [27]:
inputs = tokenizer("Explain what is a transformer architecture.", return_tensors="pt", max_length=100)
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Explain what is a transformer architecture. In this case, the input matrix X (the number of elements in its vector) and the output matrix Y are both matrices with each row containing zerosed data. The inputs to an embedding layer can be any text or image files stored on your device's storage media. As such images may contain specific information about human behaviour, they will not always contain all relevant facts about humans being violent against women. It is important that these keys used in the training process ensure that they do not falsely indicate the model


## 5. Évaluation des résultats 

Vous pouvez réutiliser le code d'évaluation rendu disponible pour la tâche 1.